<a href="https://colab.research.google.com/github/satish308/ai-mentor-portfolio./blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U ddgs


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.1 MB/s eta 0:00:00


In [ ]:
!pip install -U langchain langchain-community duckduckgo-search ddgs


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.1
    Uninstalling langgraph-1.2.1:
      Successfully uninstalled langgraph-1.2.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.1
    Uninstalling langchain-1.3.1:
      Successfully uninstalled langchain-1.3.1


In [ ]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

In [ ]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

Aug 8, 2025 ... TCS BPS Hiring - Batch of 2026. TCS BPS is bringing inspiring entry-level opportunities for Arts and Commerce graduates. As part of Business Processing ... Feb 17, 2026 ... TCS NQT Hiring Announced 2026,2025,2024. 11K views · 3 months ago ...more. Ashish Kumar. 123K. Subscribe. 363. Share. Save. Report. Comments. Mar 11, 2026 ... Tata Consultancy Services (TCS) has officially annou


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_5339/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [ ]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': '8be1d115-e384-41af-8b08-621816f08f49', 'type': 'tool_call'}]

[2] ToolMessage
    Content: Tata Consultancy Services (TCS), India's leading information technology company, has already offered jobs to 25,000 fresh graduates for the financial year 2027 (FY27). How to Apply for TCS NQT 2026? All interested and eligible candidates can register online before 14 June 2026 through the official T

[3] AIMessage
    Content: [{'type': 'text', 'text': 'I couldn\'t find a specific "2026 hiring quota" for TCS in the search results. However, I found that TCS has already offered jobs to 25,000 fresh graduates for the financial year 2027 (FY27). Additionally, the application deadline for TCS NQT 2026 is June 14, 2026.', 'extr


In [ ]:
# Pass a question that should fail the tool
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    [{'type': 'text', 'text': 'I am sorry, I cannot fulfill this request. I do not have the ability to browse to a URL and tell you what it says. I can only perform web searches based on a query.', 'extras': {'signature': 'Cp4HAQw51se61hehB2hPwNqVwc5zVo8GIK6PlHFByqDCw9oj8C5aF7qLB+yJ37Q6J+EAoDXe4h47Uhc1F
